# 🔒 Notebook 6: Security and Abuse Prevention

Direct uploads are powerful but need guardrails. Learn to prevent abuse while keeping things fast.

## Learning Objectives

By the end of this notebook, you'll understand:
- The quarantine pattern
- Content validation strategies
- Rate limiting uploads
- Common attack vectors

In [1]:
import boto3
from botocore.config import Config
import psycopg2
import hashlib
import uuid
from datetime import datetime, timedelta
from typing import Optional

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="largeblobs", user="postgres", password="postgres"
)
conn.autocommit = True

QUARANTINE_BUCKET = 'quarantine'
PROCESSED_BUCKET = 'processed'

print("✅ Connected to MinIO and PostgreSQL!")

✅ Connected to MinIO and PostgreSQL!


## 🛡️ The Quarantine Pattern

In [2]:
print("🛡️ The Quarantine Pattern")
print("=" * 60)
print("""
PROBLEM: User uploads malicious content, shares link immediately
─────────────────────────────────────────────────────────────

Without quarantine:
    Upload → Immediately accessible → Damage done!

─────────────────────────────────────────────────────────────

SOLUTION: Upload to quarantine, verify, then promote

┌────────┐    ┌─────────────┐    ┌────────────┐    ┌──────────┐
│ Upload │───>│ Quarantine  │───>│ Validation │───>│ Processed│
│        │    │   Bucket    │    │  Pipeline  │    │  Bucket  │
└────────┘    └─────────────┘    └────────────┘    └──────────┘
                    │                  │
                    │                  ▼
               Not accessible    If fails:
               until validated   Delete + notify

VALIDATION CHECKS:
• Virus/malware scan
• Content type verification (is it really an image?)
• NSFW/inappropriate content detection
• File size within limits
• Metadata extraction and validation
""")

🛡️ The Quarantine Pattern

PROBLEM: User uploads malicious content, shares link immediately
─────────────────────────────────────────────────────────────

Without quarantine:
    Upload → Immediately accessible → Damage done!

─────────────────────────────────────────────────────────────

SOLUTION: Upload to quarantine, verify, then promote

┌────────┐    ┌─────────────┐    ┌────────────┐    ┌──────────┐
│ Upload │───>│ Quarantine  │───>│ Validation │───>│ Processed│
│        │    │   Bucket    │    │  Pipeline  │    │  Bucket  │
└────────┘    └─────────────┘    └────────────┘    └──────────┘
                    │                  │
                    │                  ▼
               Not accessible    If fails:
               until validated   Delete + notify

VALIDATION CHECKS:
• Virus/malware scan
• Content type verification (is it really an image?)
• NSFW/inappropriate content detection
• File size within limits
• Metadata extraction and validation



In [3]:
class QuarantineUploadService:
    def __init__(self, s3_client, quarantine_bucket: str, processed_bucket: str):
        self.s3 = s3_client
        self.quarantine = quarantine_bucket
        self.processed = processed_bucket
    
    def generate_quarantine_url(self, user_id: str, filename: str) -> dict:
        file_id = str(uuid.uuid4())
        storage_key = f"{user_id}/{file_id}/{filename}"
        
        url = self.s3.generate_presigned_url(
            'put_object',
            Params={'Bucket': self.quarantine, 'Key': storage_key},
            ExpiresIn=3600
        )
        
        return {
            'file_id': file_id,
            'upload_url': url,
            'storage_key': storage_key,
            'status': 'quarantined'
        }
    
    def validate_file(self, storage_key: str) -> dict:
        try:
            head = self.s3.head_object(Bucket=self.quarantine, Key=storage_key)
            size = head['ContentLength']
            content_type = head.get('ContentType', 'unknown')
            
            issues = []
            
            if size > 100 * 1024 * 1024:
                issues.append("File exceeds 100MB limit")
            
            if size == 0:
                issues.append("File is empty")
            
            return {
                'valid': len(issues) == 0,
                'issues': issues,
                'size': size,
                'content_type': content_type
            }
            
        except Exception as e:
            return {'valid': False, 'issues': [str(e)]}
    
    def promote_to_processed(self, storage_key: str) -> bool:
        try:
            copy_source = {'Bucket': self.quarantine, 'Key': storage_key}
            self.s3.copy_object(
                CopySource=copy_source,
                Bucket=self.processed,
                Key=storage_key
            )
            
            self.s3.delete_object(Bucket=self.quarantine, Key=storage_key)
            return True
            
        except Exception:
            return False
    
    def reject_file(self, storage_key: str):
        self.s3.delete_object(Bucket=self.quarantine, Key=storage_key)

service = QuarantineUploadService(s3, QUARANTINE_BUCKET, PROCESSED_BUCKET)
print("✅ QuarantineUploadService ready!")

✅ QuarantineUploadService ready!


In [4]:
print("🛡️ Quarantine Upload Flow Demo")
print("=" * 60)

print("\n1️⃣ Get upload URL (to quarantine bucket)...")
upload_info = service.generate_quarantine_url('user123', 'profile.jpg')
print(f"   File ID: {upload_info['file_id']}")
print(f"   Status: {upload_info['status']}")

print("\n2️⃣ Upload file to quarantine...")
import requests
response = requests.put(
    upload_info['upload_url'],
    data=b"Fake image content for demo" * 100,
    headers={'Content-Type': 'image/jpeg'}
)
print(f"   Upload status: {response.status_code}")

print("\n3️⃣ Validate file...")
validation = service.validate_file(upload_info['storage_key'])
print(f"   Valid: {'✅ Yes' if validation['valid'] else '❌ No'}")
print(f"   Size: {validation.get('size', 0)} bytes")
print(f"   Issues: {validation.get('issues', [])}")

if validation['valid']:
    print("\n4️⃣ Promoting to processed bucket...")
    promoted = service.promote_to_processed(upload_info['storage_key'])
    print(f"   Promoted: {'✅ Yes' if promoted else '❌ No'}")
    print("\n✅ File is now accessible to users!")
else:
    print("\n4️⃣ Rejecting file...")
    service.reject_file(upload_info['storage_key'])
    print("   ❌ File rejected and deleted")

🛡️ Quarantine Upload Flow Demo

1️⃣ Get upload URL (to quarantine bucket)...
   File ID: ff9a6f0d-2d35-4549-a2b9-f256e065faf1
   Status: quarantined

2️⃣ Upload file to quarantine...
   Upload status: 200

3️⃣ Validate file...
   Valid: ✅ Yes
   Size: 2700 bytes
   Issues: []

4️⃣ Promoting to processed bucket...
   Promoted: ✅ Yes

✅ File is now accessible to users!


## 📊 Rate Limiting Uploads

In [5]:
print("📊 Rate Limiting Strategies")
print("=" * 60)
print("""
PROBLEM: Attacker requests 1000 presigned URLs, uploads 1TB
─────────────────────────────────────────────────────────────

SOLUTION: Rate limit at multiple levels

1. URL GENERATION RATE LIMIT
   • Max 10 upload URLs per user per minute
   • Prevents spam requests

2. CONCURRENT UPLOAD LIMIT
   • Max 5 pending uploads per user
   • Must complete before requesting more

3. STORAGE QUOTA
   • Max 10GB per user
   • Check before generating URL

4. FILE SIZE LIMIT (in presigned URL)
   • Bake max size into URL conditions
   • Storage rejects if exceeded
""")

📊 Rate Limiting Strategies

PROBLEM: Attacker requests 1000 presigned URLs, uploads 1TB
─────────────────────────────────────────────────────────────

SOLUTION: Rate limit at multiple levels

1. URL GENERATION RATE LIMIT
   • Max 10 upload URLs per user per minute
   • Prevents spam requests

2. CONCURRENT UPLOAD LIMIT
   • Max 5 pending uploads per user
   • Must complete before requesting more

3. STORAGE QUOTA
   • Max 10GB per user
   • Check before generating URL

4. FILE SIZE LIMIT (in presigned URL)
   • Bake max size into URL conditions
   • Storage rejects if exceeded



In [6]:
class RateLimitedUploadService:
    def __init__(self, conn, s3_client, bucket: str):
        self.conn = conn
        self.s3 = s3_client
        self.bucket = bucket
        
        self.max_pending = 5
        self.max_storage_bytes = 10 * 1024 * 1024 * 1024
        self.max_file_size = 100 * 1024 * 1024
    
    def check_rate_limits(self, user_id: str) -> dict:
        cursor = self.conn.cursor()
        
        cursor.execute("""
            SELECT COUNT(*) FROM files 
            WHERE user_id = %s AND status = 'pending'
        """, (user_id,))
        pending_count = cursor.fetchone()[0]
        
        cursor.execute("""
            SELECT COALESCE(SUM(size_bytes), 0) FROM files 
            WHERE user_id = %s AND status = 'completed'
        """, (user_id,))
        total_storage = cursor.fetchone()[0]
        
        cursor.close()
        
        issues = []
        if pending_count >= self.max_pending:
            issues.append(f"Too many pending uploads ({pending_count}/{self.max_pending})")
        
        if total_storage >= self.max_storage_bytes:
            issues.append(f"Storage quota exceeded ({total_storage / 1024 / 1024 / 1024:.1f}GB)")
        
        return {
            'allowed': len(issues) == 0,
            'issues': issues,
            'pending_uploads': pending_count,
            'storage_used_gb': total_storage / 1024 / 1024 / 1024,
            'storage_remaining_gb': (self.max_storage_bytes - total_storage) / 1024 / 1024 / 1024
        }
    
    def request_upload(self, user_id: str, filename: str, size_bytes: int) -> dict:
        limits = self.check_rate_limits(user_id)
        
        if not limits['allowed']:
            return {'error': 'Rate limited', 'details': limits['issues']}
        
        if size_bytes > self.max_file_size:
            return {'error': f'File too large (max {self.max_file_size / 1024 / 1024:.0f}MB)'}
        
        file_id = str(uuid.uuid4())
        storage_key = f"{user_id}/{file_id}/{filename}"
        
        cursor = self.conn.cursor()
        cursor.execute("""
            INSERT INTO files (id, user_id, filename, size_bytes, storage_key, status)
            VALUES (%s, %s, %s, %s, %s, 'pending')
        """, (file_id, user_id, filename, size_bytes, storage_key))
        cursor.close()
        
        conditions = [
            ["content-length-range", 0, self.max_file_size],
        ]
        
        presigned = self.s3.generate_presigned_post(
            Bucket=self.bucket,
            Key=storage_key,
            Conditions=conditions,
            ExpiresIn=3600
        )
        
        return {
            'file_id': file_id,
            'upload_url': presigned['url'],
            'fields': presigned['fields'],
            'limits': limits
        }

rate_limited_service = RateLimitedUploadService(conn, s3, 'uploads')
print("✅ RateLimitedUploadService ready!")

✅ RateLimitedUploadService ready!


In [7]:
print("📊 Rate Limiting Demo")
print("=" * 60)

test_user = '22222222-2222-2222-2222-222222222222'

print("\n1️⃣ Check limits for new user...")
limits = rate_limited_service.check_rate_limits(test_user)
print(f"   Allowed: {'✅ Yes' if limits['allowed'] else '❌ No'}")
print(f"   Pending uploads: {limits['pending_uploads']}/{rate_limited_service.max_pending}")
print(f"   Storage used: {limits['storage_used_gb']:.2f}GB")

print("\n2️⃣ Request several uploads...")
for i in range(6):
    result = rate_limited_service.request_upload(test_user, f'file_{i}.jpg', 1024000)
    if 'error' in result:
        print(f"   Upload {i+1}: ❌ {result['error']}")
        if 'details' in result:
            for detail in result['details']:
                print(f"             {detail}")
    else:
        print(f"   Upload {i+1}: ✅ URL generated (pending: {result['limits']['pending_uploads']+1})")

print("\n3️⃣ Try oversized file...")
result = rate_limited_service.request_upload(test_user, 'huge.mp4', 500 * 1024 * 1024)
if 'error' in result:
    print(f"   ❌ {result['error']}")

📊 Rate Limiting Demo

1️⃣ Check limits for new user...
   Allowed: ❌ No
   Pending uploads: 5/5
   Storage used: 0.00GB

2️⃣ Request several uploads...
   Upload 1: ❌ Rate limited
             Too many pending uploads (5/5)
   Upload 2: ❌ Rate limited
             Too many pending uploads (5/5)
   Upload 3: ❌ Rate limited
             Too many pending uploads (5/5)
   Upload 4: ❌ Rate limited
             Too many pending uploads (5/5)
   Upload 5: ❌ Rate limited
             Too many pending uploads (5/5)
   Upload 6: ❌ Rate limited
             Too many pending uploads (5/5)

3️⃣ Try oversized file...
   ❌ Rate limited


## ⚠️ Common Attack Vectors

In [8]:
print("⚠️ Common Attack Vectors")
print("=" * 60)
print("""
1. STORAGE BOMB
─────────────────────────────────────────────────────────────
   Attack: Request 1000 URLs, upload 1TB of garbage
   Defense: Rate limit URLs, storage quotas, file size in URL

2. CONTENT TYPE SPOOFING
─────────────────────────────────────────────────────────────
   Attack: Upload malware.exe as "image.jpg"
   Defense: Verify content-type server-side, magic byte check

3. PATH TRAVERSAL
─────────────────────────────────────────────────────────────
   Attack: Request URL for "../../../etc/passwd"
   Defense: Sanitize filenames, use UUIDs for storage keys

4. PRESIGNED URL REUSE
─────────────────────────────────────────────────────────────
   Attack: Share presigned URL, multiple people upload
   Defense: Short expiry, one-time tokens, IP binding

5. INCOMPLETE MULTIPART ABUSE
─────────────────────────────────────────────────────────────
   Attack: Start 1000 multipart uploads, never complete
   Defense: Lifecycle rules to auto-delete after 24h
""")

⚠️ Common Attack Vectors

1. STORAGE BOMB
─────────────────────────────────────────────────────────────
   Attack: Request 1000 URLs, upload 1TB of garbage
   Defense: Rate limit URLs, storage quotas, file size in URL

2. CONTENT TYPE SPOOFING
─────────────────────────────────────────────────────────────
   Attack: Upload malware.exe as "image.jpg"
   Defense: Verify content-type server-side, magic byte check

3. PATH TRAVERSAL
─────────────────────────────────────────────────────────────
   Attack: Request URL for "../../../etc/passwd"
   Defense: Sanitize filenames, use UUIDs for storage keys

4. PRESIGNED URL REUSE
─────────────────────────────────────────────────────────────
   Attack: Share presigned URL, multiple people upload
   Defense: Short expiry, one-time tokens, IP binding

5. INCOMPLETE MULTIPART ABUSE
─────────────────────────────────────────────────────────────
   Attack: Start 1000 multipart uploads, never complete
   Defense: Lifecycle rules to auto-delete after 24h



In [9]:
def sanitize_filename(filename: str) -> str:
    dangerous_chars = ['/', '\\', '..', '\0', '~']
    safe_name = filename
    
    for char in dangerous_chars:
        safe_name = safe_name.replace(char, '_')
    
    safe_name = ''.join(c for c in safe_name if c.isalnum() or c in '._-')
    
    if len(safe_name) > 100:
        name, ext = safe_name.rsplit('.', 1) if '.' in safe_name else (safe_name, '')
        safe_name = name[:95] + '.' + ext if ext else name[:100]
    
    return safe_name or 'unnamed'

print("🔒 Filename Sanitization Demo")
print("=" * 60)

test_filenames = [
    "normal_file.jpg",
    "../../../etc/passwd",
    "file\x00name.jpg",
    "~/secret/file.jpg",
    "a" * 200 + ".jpg",
    "<script>alert(1)</script>.jpg"
]

print("\n📝 Input → Output:")
for filename in test_filenames:
    safe = sanitize_filename(filename)
    display_input = filename[:30] + "..." if len(filename) > 30 else filename
    print(f"   {display_input:35} → {safe}")

🔒 Filename Sanitization Demo

📝 Input → Output:
   normal_file.jpg                     → normal_file.jpg
   ../../../etc/passwd                 → ______etc_passwd
   file name.jpg                       → file_name.jpg
   ~/secret/file.jpg                   → __secret_file.jpg
   aaaaaaaaaaaaaaaaaaaaaaaaaaaaaa...   → aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa.jpg
   <script>alert(1)</script>.jpg       → scriptalert1_script.jpg


## 🔬 Magic-Byte Content Validation

The client told us `Content-Type: image/jpeg` — but we can't trust that.
A real JPEG **starts with the bytes `FF D8 FF`**. Every real file type has
a similar fingerprint at the start called the "magic number".

After a direct upload, we download the first few bytes from storage
(cheap with a Range request!) and verify the fingerprint matches what
the user claimed. This catches a classic attack: uploading `malware.exe`
renamed to `cat.jpg`.


In [10]:
MAGIC_BYTES = {
    "image/jpeg": [b"\xff\xd8\xff"],
    "image/png":  [b"\x89PNG\r\n\x1a\n"],
    "image/gif":  [b"GIF87a", b"GIF89a"],
    "application/pdf": [b"%PDF-"],
    "application/zip": [b"PK\x03\x04"],  # also .docx, .xlsx, .jar
}

def detect_type(first_bytes: bytes) -> str:
    for mime, signatures in MAGIC_BYTES.items():
        if any(first_bytes.startswith(sig) for sig in signatures):
            return mime
    return "unknown"

def validate_magic_bytes(bucket: str, key: str, claimed_type: str) -> dict:
    """Download only the first 16 bytes and check the real file type."""
    resp = s3.get_object(Bucket=bucket, Key=key, Range="bytes=0-15")
    head = resp["Body"].read()
    detected = detect_type(head)
    return {
        "claimed": claimed_type,
        "detected": detected,
        "match": detected == claimed_type,
        "first_bytes_hex": head.hex(),
    }

print("🔬 Magic-Byte Validation Demo")
print("=" * 60)

# Upload three files with wrong/right claims.
samples = {
    "real.png":     (b"\x89PNG\r\n\x1a\n" + b"rest of png...", "image/png"),
    "real.jpg":     (b"\xff\xd8\xff\xe0" + b"rest of jpg...",   "image/jpeg"),
    "fake.jpg":     (b"MZ\x90\x00" + b"...actually a Windows exe!", "image/jpeg"),
}

for name, (content, claimed) in samples.items():
    key = f"validation/{name}"
    s3.put_object(Bucket=QUARANTINE_BUCKET, Key=key, Body=content)
    result = validate_magic_bytes(QUARANTINE_BUCKET, key, claimed)
    status = "✅ legit" if result["match"] else "🚨 REJECT"
    print(f"\n   {status}  {name}")
    print(f"      claimed : {result['claimed']}")
    print(f"      detected: {result['detected']}")
    print(f"      bytes[0:8]: {result['first_bytes_hex'][:16]}")

print("\n💡 Why this is cheap: Range 0-15 pulls 16 bytes, not the whole file.")
print("   For a 2GB video, we pay for 16 bytes of egress — essentially free.")


🔬 Magic-Byte Validation Demo

   ✅ legit  real.png
      claimed : image/png
      detected: image/png
      bytes[0:8]: 89504e470d0a1a0a

   ✅ legit  real.jpg
      claimed : image/jpeg
      detected: image/jpeg
      bytes[0:8]: ffd8ffe072657374

   🚨 REJECT  fake.jpg
      claimed : image/jpeg
      detected: unknown
      bytes[0:8]: 4d5a90002e2e2e61

💡 Why this is cheap: Range 0-15 pulls 16 bytes, not the whole file.
   For a 2GB video, we pay for 16 bytes of egress — essentially free.


## 🧪 Quick Quiz

1. **Why use a quarantine bucket instead of validating before upload?**

2. **What's the risk of long-lived presigned URLs?**

3. **How do storage quotas help with security?**

In [11]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why quarantine bucket:")
print("   - Can't validate before direct upload")
print("   - Bytes go client → storage (not server)")
print("   - Quarantine = validate async after upload")
print()
print("2. Long-lived URL risks:")
print("   - More time for URL to leak")
print("   - Anyone can use it until expiry")
print("   - Recommendation: 5-60 min for most cases")
print()
print("3. Storage quotas help by:")
print("   - Limiting blast radius per user")
print("   - Preventing cost explosion attacks")
print("   - Making abuse economically unviable")

📝 Quiz Answers

1. Why quarantine bucket:
   - Can't validate before direct upload
   - Bytes go client → storage (not server)
   - Quarantine = validate async after upload

2. Long-lived URL risks:
   - More time for URL to leak
   - Anyone can use it until expiry
   - Recommendation: 5-60 min for most cases

3. Storage quotas help by:
   - Limiting blast radius per user
   - Preventing cost explosion attacks
   - Making abuse economically unviable


## 📚 Summary

### Key Takeaways

1. **Quarantine pattern** - Upload to isolation, validate, then promote
2. **Rate limiting** - Limit URLs, concurrent uploads, and storage
3. **Bake limits into URLs** - Storage enforces what server can't
4. **Sanitize inputs** - Never trust user-provided filenames
5. **Cleanup incomplete** - Lifecycle rules for abandoned uploads

### 🎉 Pattern Complete!

You've learned the core strategies for handling large blobs:
1. **Direct upload** - Presigned URLs bypass your servers
2. **Resumable uploads** - Multipart for large files
3. **State sync** - Keep DB and storage consistent
4. **Fast downloads** - Range requests, parallel, CDN
5. **Security** - Quarantine, rate limits, validation

### Decision Framework

```
File size?
├── < 10MB → Normal API upload
└── > 10MB → Presigned URLs
    │
    ├── > 100MB? → Multipart/resumable
    │
    ├── User content? → Quarantine + validation
    │
    └── Frequent downloads? → CDN distribution
```